# Scraping a news website for headlines

This code scrapes information from Le Monde's website (https://www.lemonde.fr/en/) to create a csv with each article on the homepages' title, subhead, article URL, whether it's premium or not, byline, article type and image URL. 

## Importing the necessary packages and data

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

Getting the homepage data and saving it in a variable called doc

In [2]:
response = requests.get("https://www.lemonde.fr/en/")
doc = BeautifulSoup(response.text, 'html.parser')

## Searching the webpage data for the elements I need

Based on using the inspect tool on the website, I found the element div.article.article--headlines is being used to display several articles on the page. But that doesn't look like it will work for everything as different articles display on the page different. Thus I started by at any element with the article class.

In [3]:
stories = doc.find_all(class_="article")
stories

[<div class="article article--main" data-zone="titles title_1"> <div class="lmd-link-clickarea"> <span class="icon__premium icon--outside" id="zone1-premium-97249"><span class="sr-only">Subscribers only</span></span><h1 class="article__title"><a aria-describedby="zone1-premium-97249" class="lmd-link-clickarea__link" href="https://www.lemonde.fr/en/international/article/2026/07/09/trump-threatens-praises-and-reassures-allies-at-nato-summit_6755299_4.html"> <p class="article__title-label">Trump threatens, praises and reassures allies at NATO summit</p> </a> </h1> <div class="article__media-container"> <picture class="article__media"> <img alt="NATO Secretary General Mark Rutte, US President Donald Trump and British Prime Minister Keir Starmer, in Ankara, July 8, 2026." height="2" loading="eager" sizes="(min-width: 1024px) 421px, 100vw" src="https://img.lemde.fr/2026/07/08/0/0/5097/3398/400/266/75/0/8c1d9f0_ftp-1-p1aqeb6zhjzj-2026-07-08t104306z-1833425710-rc2m9ma37b01-rtrmadp-3-nato-summi

#### First attempt at exploring the data using Soma's example code from class

In [4]:
# Starting off without ANY rows
rows = []

for story in stories:
    print("----")
    # Starting off knowing NONE of the columns of data for this datapoint?
    row = {}

    try:
        row['title'] = story.select_one('h1, h2, h3').text.strip()
    except:
        print("Headline not found!")
        continue
        
    try:
        # Find me a media__link OR a reel_link
        row['href'] = story.select_one('a')['href']
    except:
        print("Couldn't find a link")

    try:
        row['tag'] = story.select_one('.ewmvhB').text.strip()
    except:
        print("Couldn't find a tag!")

    try:
        row['tag'] = story.find(attrs={'data-testid': 'card-description'}).text.strip()
    except:
        print("Couldn't find a summary")

    print(row)
    # When we're done adding info to our row, we're going to add it into our list
    # of rows
    rows.append(row)

----
Couldn't find a tag!
Couldn't find a summary
{'title': 'Trump threatens, praises and reassures allies at NATO summit', 'href': 'https://www.lemonde.fr/en/international/article/2026/07/09/trump-threatens-praises-and-reassures-allies-at-nato-summit_6755299_4.html'}
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!
----
Headline not found!


That did not work at all. So I will start exploring the data on my own.

#### Exploring the first 3 and last 3 stories myself just to get a sense of what elements are in it

In [5]:
for story in stories[:3]:
    print("----")
    print(story)

print(" ----- last 3 ----- ")

for story in stories[(len(stories)-3):]:
    print("----")
    print(story)

----
<div class="article article--main" data-zone="titles title_1"> <div class="lmd-link-clickarea"> <span class="icon__premium icon--outside" id="zone1-premium-97249"><span class="sr-only">Subscribers only</span></span><h1 class="article__title"><a aria-describedby="zone1-premium-97249" class="lmd-link-clickarea__link" href="https://www.lemonde.fr/en/international/article/2026/07/09/trump-threatens-praises-and-reassures-allies-at-nato-summit_6755299_4.html"> <p class="article__title-label">Trump threatens, praises and reassures allies at NATO summit</p> </a> </h1> <div class="article__media-container"> <picture class="article__media"> <img alt="NATO Secretary General Mark Rutte, US President Donald Trump and British Prime Minister Keir Starmer, in Ankara, July 8, 2026." height="2" loading="eager" sizes="(min-width: 1024px) 421px, 100vw" src="https://img.lemde.fr/2026/07/08/0/0/5097/3398/400/266/75/0/8c1d9f0_ftp-1-p1aqeb6zhjzj-2026-07-08t104306z-1833425710-rc2m9ma37b01-rtrmadp-3-nato-s

## Building a scraper to capture the information I need

We are looking for...<br>
headline = article__title class <br>
subtitle = article__desc class (only exists for some headlines) <br>
article url = lmd-link-clickarea__link class <br>
image url = src attribute withing the image tag <br>
bylines = article__byline class (but most aren't shown on the homepage) another option I have seen is article__author-name <br>
whether it is premium or not = sr-only class <br>
article type = article__type class (doesn't exist for all stories) <br>

This code uses the above information in a for loop to build a data set with the information we need. I am printing the information for each step to make sure it is working as planned.

In [6]:
rows = []

for story in stories:
    print("----")
    # Starting off knowing NONE of the columns of data for this datapoint?
    row = {}

    try:
        # This gets the headlines generally but will need to be cleaned up using pandas
        row["headline"] = (story.select_one(".article__title").get_text())
        print(row["headline"])
    except:
        print("Headline not found!")
        continue
        
    try:
        # finding the subtitles
        row["subtitle"] = (story.select_one(".article__desc").get_text())
        print(row["subtitle"])
    except:
        print("Couldn't find a subtitle")

    # The .lmd-link-clickarea__link is not getting the article urls for everything unfortunately
    # so I have added one more attempt to catch stragglers before instructing the loop to give up
    try:
        row["article_url"] = story.select_one(".lmd-link-clickarea__link")["href"]
        print(row["article_url"])
    except:
        try:
            row["article_url"] = story.select_one("a")["href"]
            print(row["article_url"])
        except:
            print("Couldn't find an article url")

    try:
        row["image_url"] = story.select_one("img")["src"]
        print(row["image_url"])
    except:
        print("Couldn't find an image_url")
    
    # Bylines appear under different names and many of the articles do not have bylines at all on the homepage
    try:
        row["byline"] = (story.select_one(".article__byline").get_text())
        print(f"byline {row["byline"]}")
    except:
        try:
            row["byline"] = (story.select_one(".article__author-name").get_text())
            print(f"author name {row["byline"]}")
        except:
            print("Couldn't find a byline")

    try:
        if story.select_one(".sr-only") != None:
            row["paywall_status"] = "subscriber only"
            print(row["paywall_status"])
    except:
        print("Couldn't find subscriber status")

    try:
        row["article_type"] = (story.select_one(".article__type").get_text())
        print(row["article_type"])
    except:
        print("Couldn't find article type")
        
    # # When we're done adding info to our row, we're going to add it into our list    
    print(row)
    rows.append(row)

----
 Trump threatens, praises and reassures allies at NATO summit  
In Turkey, the American president, after criticizing Spain and reiterating his ambitions regarding Greenland, reaffirmed his commitment to the alliance's mutual defense clause and gave Ukraine the green light to produce Patriot missiles under license.
https://www.lemonde.fr/en/international/article/2026/07/09/trump-threatens-praises-and-reassures-allies-at-nato-summit_6755299_4.html
https://img.lemde.fr/2026/07/08/0/0/5097/3398/400/266/75/0/8c1d9f0_ftp-1-p1aqeb6zhjzj-2026-07-08t104306z-1833425710-rc2m9ma37b01-rtrmadp-3-nato-summit-trump.JPG
Couldn't find a byline
subscriber only
Couldn't find article type
{'headline': ' Trump threatens, praises and reassures allies at NATO summit  ', 'subtitle': "In Turkey, the American president, after criticizing Spain and reiterating his ambitions regarding Greenland, reaffirmed his commitment to the alliance's mutual defense clause and gave Ukraine the green light to produce Patri

## Exporting the results

Saving it as a data frame and having a quick glance

In [7]:
df = pd.DataFrame(rows)
df

,headline,subtitle,article_url,image_url,paywall_status,byline,article_type
0,"Trump threatens, praises and reassures allies...","In Turkey, the American president, after criti...",https://www.lemonde.fr/en/international/articl...,https://img.lemde.fr/2026/07/08/0/0/5097/3398/...,subscriber only,NaN,NaN
1,Renewed US-Iran hostilities highlight Tehran's...,NaN,https://www.lemonde.fr/en/international/articl...,https://img.lemde.fr/2026/07/06/3/0/4432/2954/...,subscriber only,NaN,NaN
2,Le Pen cuts short first presidential campaign ...,NaN,https://www.lemonde.fr/en/politics/article/202...,https://img.lemde.fr/2026/07/08/0/0/4723/3149/...,subscriber only,NaN,NaN
3,"Graham Platner, Democratic candidate for US Se...","Platner, the Democratic Senate nominee for Mai...",https://www.lemonde.fr/en/international/articl...,https://img.lemde.fr/2026/06/10/0/0/4279/2853/...,NaN,NaN,NaN
4,EWC 2026 in Paris: Behind the scenes of the co...,By hosting this Saudi video game tournament in...,https://www.lemonde.fr/en/pixels/article/2026/...,https://img.lemde.fr/2023/06/16/0/0/6000/4000/...,subscriber only,NaN,NaN
5,France vs. Morocco: The Atlas Lions seek redem...,The Moroccan national football team has only b...,https://www.lemonde.fr/en/le-monde-africa/arti...,https://img.lemde.fr/2026/07/04/0/0/4327/2885/...,subscriber only,NaN,NaN
6,"How Ukraine is turning Crimea, the Russians' R...","The annexed Ukrainian peninsula, which Vladimi...",https://www.lemonde.fr/en/international/articl...,https://img.lemde.fr/2026/06/10/0/200/2160/144...,subscriber only,Faustine Vincent and Marie Jégo,In Depth
7,Western Europe experiences hottest June on record,NaN,https://www.lemonde.fr/en/international/articl...,https://img.lemde.fr/2026/06/25/1/0/7270/4846/...,NaN,NaN,NaN
8,US judge orders Trump to pay $5 million he owe...,NaN,https://www.lemonde.fr/en/international/articl...,https://img.lemde.fr/2026/07/08/96/0/4451/2967...,NaN,NaN,NaN
9,Russia to continue providing military support ...,NaN,https://www.lemonde.fr/en/international/articl...,https://img.lemde.fr/2026/07/08/0/0/1279/853/1...,NaN,NaN,NaN


Exporting the data as a csv

In [8]:
df.to_csv("le_monde_daily_news.csv")

## Making the CSV auto-updating

I have created a github action to autoupdate this code, using Jonathan Soma's tutorial
https://www.youtube.com/watch?v=QNKxzkNpsko.

For future reference:
Step 1: Make your working files into a github repository
Step 2: Create a github action called update.yml using Soma's code 
Step 3: Make sure that you use a Cron code for the frequency of update that you want and that the file referenced is the correct one. Try to run the action and trouble shoot any error by checking at which point the code stopped being able to run.
